<a href="https://colab.research.google.com/github/JiviteshG/GenerativeAI/blob/dev/HuggingFace_Speech_to_Text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Speech-to-Text Transcription using Hugging Face Transformers

This notebook successfully demonstrates a complete workflow for **Speech-to-Text Transcription using Hugging Face Transformers**. The key steps covered include:

1.  **Environment Setup**: Installation of necessary libraries (`transformers`, `librosa`, `torch`).
2.  **Model Loading**: Loading the pre-trained `Wav2Vec2Tokenizer` and `Wav2Vec2ForCTC` model (`facebook/wav2vec2-base-960h`) from Hugging Face.
3.  **Audio Preprocessing**: Loading a sample audio file (`test01_20s.wav`) using `librosa` and ensuring it's resampled to the required 16kHz.
4.  **Inference**: Processing the audio through the tokenizer to obtain `input_values`, feeding them to the model to get `logits`, deriving `predicted_ids` using `argmax`, and finally decoding these IDs back into a human-readable `transcriptions` string.
5.  **Output Saving**: The final transcription was saved to a text file named `transcription.txt`.



In [9]:
pip install transformers


In [10]:
from transformers import pipeline # pipeline defines flow of data from origin to destination system ans how to transform it

In [11]:
import transformers
print(transformers.__version__)

4.57.3


In [12]:
import librosa # audio and music analysis package in python
import torch # pythorch - comp viosion NLP, developed my meta and owned by lixun
import IPython.display as display # intearctive commanline terminal for python and webbased notbook platform
from transformers import Wav2Vec2ForCTC, Wav2Vec2Tokenizer # Wav2Vec2ForCTC: how to finetune a speech recognition model,
import numpy as np # working with arrays, linear algenra, fourier transforms

In [13]:
tokenizer = Wav2Vec2Tokenizer.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Wav2Vec2CTCTokenizer'. 
The class this function is called from is 'Wav2Vec2Tokenizer'.
/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/tokenization_wav2vec2.py:720: FutureWarning: The class `Wav2Vec2Tokenizer` is deprecated and will be removed in ver

In [14]:
# Download a sample mp4 file and rename it to v.mp4
# !wget -O v.mp4 http://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerJoyrides.mp4
# !curl -L -o v.mp4 "https://github.com/rafaelreis-hotmart/Audio-Sample-Files/raw/master/sample.mp4"
# !curl -L -o v.mp3 "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3"
# !curl -L -o v.mp4 "https://www.archive.org/download/short_speech_samples/male_speech_sample.mp4"
!curl -L -o v3.mp4 "https://www.archive.org/download/short_speech_samples/female_speech_sample.mp4"
# !curl -L -o v3.mp4 "https://archive.org/download/Apollo11Audio/40-06103.mp4"
# -L follows redirects, -o names the output file
# !curl -L -o v3.mp4 "https://www.nasa.gov/wp-content/uploads/static/history/alsj/a11/a11.landing.mp4"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   145  100   145    0     0    213      0 --:--:-- --:--:-- --:--:--   214
100  152k    0  152k    0     0  38664      0 --:--:--  0:00:04 --:--:-- 48681


In [17]:
# Extract audio from v2.mp4 and convert to a standard 16kHz WAV
!ffmpeg -i v3.mp4 -ar 16000 -ac 1 -c:a pcm_s16le v3.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [18]:
# loading audio files
audio1, sampling_rate1 = librosa.load("/content/test01_20s.wav", sr=16000) #

In [19]:
audio1, sampling_rate1

(array([-3.0517578e-05,  0.0000000e+00,  3.0517578e-05, ...,
         0.0000000e+00,  0.0000000e+00,  3.0517578e-05], dtype=float32),
 16000)

In [20]:
display.Audio(audio1, rate=sampling_rate1)

In [21]:
input_values = tokenizer(audio1, return_tensors="pt").input_values

In [22]:
input_values

tensor([[0.0006, 0.0011, 0.0017,  ..., 0.0011, 0.0011, 0.0017]])

In [23]:
logits = model(input_values).logits
logits

tensor([[[ 15.4988, -27.4559, -27.0604,  ...,  -7.2004,  -7.2163,  -8.3170],
         [ 15.4796, -27.2306, -26.8336,  ...,  -6.7958,  -6.8107,  -8.2089],
         [ 15.4920, -27.2671, -26.8681,  ...,  -6.7210,  -6.8076,  -8.1847],
         ...,
         [ 15.6017, -27.6215, -27.2329,  ...,  -7.0008,  -7.2178,  -8.1846],
         [ 15.8038, -27.7616, -27.3547,  ...,  -6.9453,  -7.2343,  -8.3626],
         [ 15.7574, -27.7355, -27.3336,  ...,  -7.0641,  -7.3827,  -8.4322]]],
       grad_fn=<ViewBackward0>)

In [31]:
predicted_ids = torch.argmax(logits, dim=-1)

In [32]:
transcriptions = tokenizer.decode(predicted_ids[0])

In [33]:
transcriptions

'DANCING IN THE MASQUERADE IDLE TRUTH AN PLAIN SIGHT JADED POP ROLL CLICK SHOT WHO WILL I BE TO DAY OR NOT BUT SUCH A TIDE AS MOVING SEEMS ASLEEP TOO FULL FOR SOUND AND POAM WHEN THAT WHICH DREW FROM OUT THE BOUNDLESS DEEP TURNS AGAIN HOME TWILIGHT AND EVENING BELL AND AFTER THAT'

### Final Transcription Result:

Below is the complete transcription generated by the Wav2Vec2 model from the input audio file:

```
DANCING IN THE MASQUERADE IDLE TRUTH AN PLAIN SIGHT JADED POP ROLL CLICK SHOT WHO WILL I BE TO DAY OR NOT BUT SUCH A TIDE AS MOVING SEEMS ASLEEP TOO FULL FOR SOUND AND POAM WHEN THAT WHICH DREW FROM OUT THE BOUNDLESS DEEP TURNS AGAIN HOME TWILIGHT AND EVENING BELL AND AFTER THAT
```

In [35]:
with open('transcription.txt', 'w') as f:
    f.write(transcriptions)
print("Transcription saved to transcription.txt")

Transcription saved to transcription.txt
